# Setup і завантаження даних

In [ ]:
import pixeltable_pgserver as pgserver # аналог pgserver
from sqlalchemy import create_engine, text
import pandas as pd

db = pgserver.get_server("./temp", cleanup_mode="delete")

uri = db.get_uri(driver="psycopg2")
print("Connection URI:", uri)
print(db.get_postmaster_info().pgdata)
print(db.get_postmaster_info().port)

engine = create_engine(uri, echo=False, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

Connection URI: postgresql+psycopg2://postgres:@/postgres?host=/Users/vladyslavratushnyi/Learning/DB_DS/hw6/temp
/Users/vladyslavratushnyi/Learning/DB_DS/hw6/temp
5432
PostgreSQL 18.4 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [2]:
# кастомна функція для виконання sql
def sql_exec(sql_quary:str, engine=engine, is_file=False, printout=False):
    try:
        if is_file:
            with open(sql_quary, 'r') as f:
                sql_quary = f.read()

        ddl_statements = [
            query.strip() + ";" 
            for query in sql_quary.split(';') 
            if query.strip()]

        with engine.begin() as conn:
            for stmt in ddl_statements:
                output = conn.execute(text(stmt))
                if output.returns_rows and printout:
                    for i in output.mappings().fetchall():
                        print(i)


        return f"{len(ddl_statements)} команд виконано"

    except Exception as e:
        print('Операція не виконана:', e)


def sql_reader(query:str, engine=engine, is_file=False)->pd.DataFrame:
    try:
        if is_file:
            with open(query, 'r') as f:
                query = f.read()

        with engine.connect() as con:
            return pd.read_sql_query(query, con)

    except Exception as e:
        print('Операція не виконана:', e)

In [3]:
sql_exec('sql_cmnds/com1.sql', is_file=True)

'3 команд виконано'

In [4]:
query = """
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id;
"""
sql_reader(query)

,pickup_zone_id,n_days
0,1,60
1,2,60
2,3,60
3,4,60
4,5,60


# Time-series ML features

In [5]:
# Lag features: lag-1d, lag-7d, lag-30d
query = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
sql_reader(query)

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-02-01,550,NaN,NaN,None
1,1,2024-02-02,587,550.0,NaN,None
2,1,2024-02-03,624,587.0,NaN,None
3,1,2024-02-04,661,624.0,NaN,None
4,1,2024-02-05,698,661.0,NaN,None
5,1,2024-02-06,585,698.0,NaN,None
6,1,2024-02-07,622,585.0,NaN,None
7,1,2024-02-08,659,622.0,550.0,None
8,1,2024-02-09,696,659.0,587.0,None
9,1,2024-02-10,733,696.0,624.0,None


In [6]:
# Rolling 7-day average fare
query = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
sql_reader(query)

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-02-01,13.2,13.200000,1
1,1,2024-02-02,13.6,13.400000,2
2,1,2024-02-03,14.0,13.600000,3
3,1,2024-02-04,14.4,13.800000,4
4,1,2024-02-05,13.2,13.680000,5
5,1,2024-02-06,13.6,13.666667,6
6,1,2024-02-07,14.0,13.714286,7
7,1,2024-02-08,14.4,13.885714,7
8,1,2024-02-09,13.2,13.828571,7
9,1,2024-02-10,13.6,13.771429,7


In [10]:
# Rolling 30-day fare volatility
query = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 15;
"""
sql_reader(query)

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-02-01,13.2,NaN,1
1,1,2024-02-02,13.6,0.282843,2
2,1,2024-02-03,14.0,0.400000,3
3,1,2024-02-04,14.4,0.516398,4
4,1,2024-02-05,13.2,0.521536,5
5,1,2024-02-06,13.6,0.467618,6
6,1,2024-02-07,14.0,0.445079,7
7,1,2024-02-08,14.4,0.478091,8
8,1,2024-02-09,13.2,0.489898,9
9,1,2024-02-10,13.6,0.463801,10


In [9]:
# Expanding cumulative trips
query = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 15;
"""
sql_reader(query)

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-02-01,550,550,550.000000
1,1,2024-02-02,587,1137,568.500000
2,1,2024-02-03,624,1761,587.000000
3,1,2024-02-04,661,2422,605.500000
4,1,2024-02-05,698,3120,624.000000
5,1,2024-02-06,585,3705,617.500000
6,1,2024-02-07,622,4327,618.142857
7,1,2024-02-08,659,4986,623.250000
8,1,2024-02-09,696,5682,631.333333
9,1,2024-02-10,733,6415,641.500000


In [ ]:
sql_exec('sql_cmnds/comb_view.sql', is_file=True)

'1 команд виконано'

In [12]:
# Combined feature view
query = """
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
sql_reader(query)

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-02,760,14.0,2370.0,873,801,550,13.714286,0.442563,22105,713.064516
1,1,2024-03-03,797,14.4,2395.0,760,838,587,13.885714,0.454049,22902,715.687500
2,1,2024-03-04,834,13.2,2420.0,797,725,624,13.828571,0.466831,23736,719.272727
3,1,2024-03-05,871,13.6,2445.0,834,762,661,13.771429,0.454049,24607,723.735294
4,1,2024-03-06,908,14.0,2470.0,871,799,698,13.714286,0.442563,25515,729.000000
5,1,2024-03-07,795,14.4,2495.0,908,836,585,13.885714,0.454049,26310,730.833333
6,1,2024-03-08,832,13.2,2520.0,795,873,622,13.828571,0.466831,27142,733.567568
7,1,2024-03-09,869,13.6,2545.0,832,760,659,13.771429,0.454049,28011,737.131579
8,1,2024-03-10,906,14.0,2570.0,869,797,696,13.714286,0.442563,28917,741.461538
9,1,2024-03-11,943,14.4,2595.0,906,834,733,13.885714,0.454049,29860,746.500000


# JSONB-таблиця для подій

In [15]:
sql_exec('sql_cmnds/jsonb_table.sql', is_file=True)

'3 команд виконано'

In [ ]:
# check the structure of the table
query = """
SELECT event_type, COUNT(*) AS n_events,
    (SELECT string_agg(k, ', ' ORDER BY k)
FROM (SELECT DISTINCT jsonb_object_keys(payload) k
    FROM t7_hw_events t2
WHERE t2.event_type = t.event_type) s) AS payload_keys
FROM t7_hw_events t
GROUP BY event_type
ORDER BY event_type;
"""
sql_reader(query)

,event_type,n_events,payload_keys
0,add_to_cart,10000,"price, product_id, quantity, tags"
1,checkout,10000,"items_count, order_id, payment_method, total"
2,click,10000,"device, element_id, page_url"
3,page_view,10000,"device, referrer, url"
4,recommendation_shown,10000,"experiment, model_name, model_version, recomme..."


In [ ]:
# JSONB Query 1. Containment @>
query = """
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'add_to_cart'
  AND payload @> jsonb_build_object('tags', jsonb_build_array('promo'))
LIMIT 10;
"""
sql_reader(query)

,event_id,user_id,event_ts,payload
0,12,13,2024-03-20 01:00:00+00:00,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
1,27,28,2024-03-05 01:00:00+00:00,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
2,42,43,2024-02-19 01:00:00+00:00,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
3,57,58,2024-02-04 01:00:00+00:00,"{'tags': ['promo'], 'price': 67.0, 'quantity':..."
4,72,73,2024-03-20 01:00:00+00:00,"{'tags': ['promo'], 'price': 82.0, 'quantity':..."
5,87,88,2024-03-05 01:00:00+00:00,"{'tags': ['promo'], 'price': 97.0, 'quantity':..."
6,102,103,2024-02-19 01:00:00+00:00,"{'tags': ['promo'], 'price': 12.0, 'quantity':..."
7,117,118,2024-02-04 01:00:00+00:00,"{'tags': ['promo'], 'price': 27.0, 'quantity':..."
8,132,133,2024-03-20 01:00:00+00:00,"{'tags': ['promo'], 'price': 42.0, 'quantity':..."
9,147,148,2024-03-05 01:00:00+00:00,"{'tags': ['promo'], 'price': 57.0, 'quantity':..."


In [20]:
# JSONB Query 2. Key existence ?
query = """
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'referrer'
LIMIT 10;
"""
sql_reader(query)

,event_id,event_type,payload
0,5,page_view,"{'url': '/product/6', 'device': 'tablet', 'ref..."
1,10,page_view,"{'url': '/product/11', 'device': 'desktop', 'r..."
2,15,page_view,"{'url': '/product/16', 'device': 'mobile', 're..."
3,20,page_view,"{'url': '/product/21', 'device': 'tablet', 're..."
4,25,page_view,"{'url': '/product/26', 'device': 'desktop', 'r..."
5,30,page_view,"{'url': '/product/31', 'device': 'mobile', 're..."
6,35,page_view,"{'url': '/product/36', 'device': 'tablet', 're..."
7,40,page_view,"{'url': '/product/41', 'device': 'desktop', 'r..."
8,45,page_view,"{'url': '/product/46', 'device': 'mobile', 're..."
9,50,page_view,"{'url': '/product/51', 'device': 'tablet', 're..."


In [21]:
# Key existence ?|
query = """
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['payment_method', 'items_count']
LIMIT 10;
"""
sql_reader(query)

,event_id,event_type,payload
0,3,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,8,checkout,"{'total': 58.0, 'order_id': 'o_100008', 'items..."
2,13,checkout,"{'total': 63.0, 'order_id': 'o_100013', 'items..."
3,18,checkout,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
4,23,checkout,"{'total': 73.0, 'order_id': 'o_100023', 'items..."
5,28,checkout,"{'total': 78.0, 'order_id': 'o_100028', 'items..."
6,33,checkout,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
7,38,checkout,"{'total': 88.0, 'order_id': 'o_100038', 'items..."
8,43,checkout,"{'total': 93.0, 'order_id': 'o_100043', 'items..."
9,48,checkout,"{'total': 98.0, 'order_id': 'o_100048', 'items..."


In [23]:
# JSONB Query 3. JSONPath
query = """
SELECT
    event_id,
    jsonb_path_query(payload, '$.tags[*]') AS tag
FROM t7_hw_events
WHERE event_type = 'add_to_cart'
LIMIT 10;
"""
sql_reader(query)

,event_id,tag
0,2,clearance
1,7,new
2,12,promo
3,17,clearance
4,22,new
5,27,promo
6,32,clearance
7,37,new
8,42,promo
9,47,clearance


In [24]:
query = """
SELECT
    event_id,
    payload ->> 'product_id' AS product_id,
    (payload ->> 'price')::NUMERIC(10, 2) AS price
FROM t7_hw_events
WHERE event_type = 'add_to_cart'
  AND jsonb_path_exists(payload, '$.price ? (@ > 80)')
LIMIT 10;
"""
sql_reader(query)

,event_id,product_id,price
0,72,73,82.0
1,77,78,87.0
2,82,83,92.0
3,87,88,97.0
4,92,93,102.0
5,97,98,107.0
6,172,173,82.0
7,177,178,87.0
8,182,183,92.0
9,187,188,97.0


# GIN-index demo + EXPLAIN ANALYZE

In [ ]:
# Containment query без GIN-індексу
query = """
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('tags', 'new');
"""
sql_exec(query, printout=True)

{'QUERY PLAN': 'Aggregate  (cost=1862.01..1862.02 rows=1 width=8) (actual time=59.201..59.202 rows=1.00 loops=1)'}
{'QUERY PLAN': '  Buffers: shared hit=1112'}
{'QUERY PLAN': '  ->  Seq Scan on t7_hw_events  (cost=0.00..1862.00 rows=3 width=0) (actual time=59.198..59.198 rows=0.00 loops=1)'}
{'QUERY PLAN': "        Filter: (payload @> jsonb_build_object('tags', 'new'))"}
{'QUERY PLAN': '        Rows Removed by Filter: 50000'}
{'QUERY PLAN': '        Buffers: shared hit=1112'}
{'QUERY PLAN': 'Planning:'}
{'QUERY PLAN': '  Buffers: shared hit=8'}
{'QUERY PLAN': 'Planning Time: 0.235 ms'}
{'QUERY PLAN': 'Execution Time: 59.224 ms'}


'5 команд виконано'

In [26]:
# GIN-індекс для containment / JSONPath workload
query = """
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;
"""
sql_exec(query, printout=True)

{'gin_index_size': '1144 kB'}


'3 команд виконано'

In [27]:
# після GIN
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
"""
sql_exec(query, printout=True)

{'QUERY PLAN': 'Aggregate  (cost=1203.46..1203.47 rows=1 width=8) (actual time=5.002..5.003 rows=1.00 loops=1)'}
{'QUERY PLAN': '  Buffers: shared hit=1115'}
{'QUERY PLAN': '  ->  Bitmap Heap Scan on t7_hw_events  (cost=34.14..1195.27 rows=3275 width=0) (actual time=0.401..4.817 rows=3334.00 loops=1)'}
{'QUERY PLAN': "        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))"}
{'QUERY PLAN': '        Heap Blocks: exact=1111'}
{'QUERY PLAN': '        Buffers: shared hit=1115'}
{'QUERY PLAN': '        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..33.33 rows=3275 width=0) (actual time=0.270..0.270 rows=3334.00 loops=1)'}
{'QUERY PLAN': "              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))"}
{'QUERY PLAN': '              Index Searches: 1'}
{'QUERY PLAN': '              Buffers: shared hit=4'}
{'QUERY PLAN': 'Planning:'}
{'QUERY PLAN': '  Buffers: shared hit=29'}
{'QUERY PLAN': 'Planning Time: 1.697 ms'}
{'QUERY

'2 команд виконано'

In [34]:
# Коли GIN не допомагає: path-specific text equality
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'desktop';
"""
sql_exec(query, printout=True)

{'QUERY PLAN': 'Aggregate  (cost=1862.62..1862.63 rows=1 width=8) (actual time=11.929..11.930 rows=1.00 loops=1)'}
{'QUERY PLAN': '  Buffers: shared hit=1112'}
{'QUERY PLAN': '  ->  Seq Scan on t7_hw_events  (cost=0.00..1862.00 rows=250 width=0) (actual time=0.017..11.315 rows=8333.00 loops=1)'}
{'QUERY PLAN': "        Filter: ((payload ->> 'device'::text) = 'desktop'::text)"}
{'QUERY PLAN': '        Rows Removed by Filter: 41667'}
{'QUERY PLAN': '        Buffers: shared hit=1112'}
{'QUERY PLAN': 'Planning Time: 0.087 ms'}
{'QUERY PLAN': 'Execution Time: 11.957 ms'}


'1 команд виконано'

In [ ]:
# with expression index
query = """
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'desktop';
"""
sql_exec(query, printout=True)

{'QUERY PLAN': 'Aggregate  (cost=1355.05..1355.06 rows=1 width=8) (actual time=2.016..2.016 rows=1.00 loops=1)'}
{'QUERY PLAN': '  Buffers: shared hit=1111 read=9'}
{'QUERY PLAN': '  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.98..1334.18 rows=8347 width=0) (actual time=0.389..1.685 rows=8333.00 loops=1)'}
{'QUERY PLAN': "        Recheck Cond: ((payload ->> 'device'::text) = 'desktop'::text)"}
{'QUERY PLAN': '        Heap Blocks: exact=1111'}
{'QUERY PLAN': '        Buffers: shared hit=1111 read=9'}
{'QUERY PLAN': '        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.89 rows=8347 width=0) (actual time=0.263..0.263 rows=8333.00 loops=1)'}
{'QUERY PLAN': "              Index Cond: ((payload ->> 'device'::text) = 'desktop'::text)"}
{'QUERY PLAN': '              Index Searches: 1'}
{'QUERY PLAN': '              Buffers: shared read=9'}
{'QUERY PLAN': 'Planning:'}
{'QUERY PLAN': '  Buffers: shared hit=5 read=1'}
{'QUERY PLAN': 'Planning Time: 0.090 ms'}
{'QUERY PLA

'3 команд виконано'

# JSONB vs normalized

In [ ]:
# creating and filling a normalized table for JSONB data
sql_exec('sql_cmnds/norm_table.sql', is_file=True, printout=True)

{'n_checkouts': 10000}


'6 команд виконано'

In [ ]:
# JSONB-версія
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
"""
sql_exec(query, printout=True)

{'QUERY PLAN': 'Limit  (cost=2061.88..2061.89 rows=3 width=72) (actual time=13.314..13.315 rows=3.00 loops=1)'}
{'QUERY PLAN': '  Buffers: shared hit=1112'}
{'QUERY PLAN': '  ->  Sort  (cost=2061.88..2075.53 rows=5460 width=72) (actual time=13.313..13.314 rows=3.00 loops=1)'}
{'QUERY PLAN': "        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC"}
{'QUERY PLAN': '        Sort Method: quicksort  Memory: 25kB'}
{'QUERY PLAN': '        Buffers: shared hit=1112'}
{'QUERY PLAN': '        ->  HashAggregate  (cost=1909.41..1991.31 rows=5460 width=72) (actual time=13.269..13.305 rows=3.00 loops=1)'}
{'QUERY PLAN': "              Group Key: (payload ->> 'payment_method'::text)"}
{'QUERY PLAN': '              Batches: 1  Memory Usage: 161kB'}
{'QUERY PLAN': '              Buffers: shared hit=1112'}
{'QUERY PLAN': '              ->  Seq Scan on t7_hw_events  (cost=0.00..1761.63 rows=9852 width=132) (actual time=0.021..8.351 rows=10000.00 loops=1)'}
{'QUERY PLAN': "                  

'1 команд виконано'

In [40]:
# Normalized-версія
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
"""
sql_exec(query, printout=True)

{'QUERY PLAN': 'Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=4.052..4.054 rows=3.00 loops=1)'}
{'QUERY PLAN': '  Buffers: shared hit=94'}
{'QUERY PLAN': '  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=4.050..4.051 rows=3.00 loops=1)'}
{'QUERY PLAN': '        Sort Key: (avg(total)) DESC'}
{'QUERY PLAN': '        Sort Method: quicksort  Memory: 25kB'}
{'QUERY PLAN': '        Buffers: shared hit=94'}
{'QUERY PLAN': '        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=4.040..4.042 rows=3.00 loops=1)'}
{'QUERY PLAN': '              Group Key: payment_method'}
{'QUERY PLAN': '              Batches: 1  Memory Usage: 32kB'}
{'QUERY PLAN': '              Buffers: shared hit=94'}
{'QUERY PLAN': '              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.007..0.876 rows=10000.00 loops=1)'}
{'QUERY PLAN': '                    Buffers: shared hit=94'}
{'QUERY PLAN': 'Planning:'}
{'QUERY PLAN': '  

'1 команд виконано'

PostgreSQL JSONB найкраще працює, коли напівструктуровані дані живуть поруч із реляційними: можна робити JOIN із нормалізованими таблицями, покладатися на ACID-транзакції, обмеження цілісності та зрілий оптимізатор. Індекси GIN дозволяють швидко шукати всередині payload.

MongoDB aggregation pipeline природний для document-first сценаріїв, де документ є основною одиницею, схема часто змінюється, а обробка йде послідовними стадіями ($match, $unwind, $group). Горизонтальне масштабування вбудоване.

Raw clickstream великого обсягу, який переважно дописується й рідко змінюється, краще зберігати в document store або data lake це дешевше, а транзакційні гарантії тут майже не потрібні.

Curated ML features зручніше тримати в типізованих SQL-таблицях: фіксовані типи, CHECK-обмеження, версіонування та передбачувані запити роблять ознаки відтворюваними й менш схильними до тихих помилок, як у нашій t7_hw_checkouts_norm.

# Reflection

1. **Плани до і після GIN.** До індексу планувальник вибрав Seq Scan: cost=1862.01, actual time=59.2 ms, Rows Removed by Filter: 50000, Buffers: shared hit=1112. Після створення jsonb_path_ops GIN-індексу Seq Scan змінився на Bitmap Index Scan + Bitmap Heap Scan. Для payment_method = 'card' cost впав до 1203.46, час до 5.0 ms, для referrer = 'google' до 1178.83 і 3.1 ms. Застереження: ці запити різні, а перший, схоже, виконувався на холодному кеші. До того ж Buffers майже не зменшились (≈1115), бо збіги розкидані по всіх блоках таблиці.

2. **Чому GIN не для ->> 'device'.** GIN індексує структуру документа, тому оператор @> він використовує напряму. Вираз payload ->> 'device' = 'desktop' для нього є звичайним порівнянням тексту, тож індекс не підходить, і ми побачили Seq Scan (11.9 ms). B-tree по виразу ((payload ->> 'device')) дав Bitmap Index Scan і 2.0 ms.

3. **Read/write trade-off.** Пошук по вмісту payload пришвидшується. Дорожчими стають INSERT та UPDATE, бо індекс треба оновлювати, а ще він займає місце (1144 kB тут) і потребує обслуговування (VACUUM, bloat).

4. **JSONB чи нормалізована таблиця.** JSONB підходить для сирих подій зі змінною схемою та ранньої розвідки даних. Нормалізована таблиця краща для стабільних, часто запитуваних полів і curated features: типи, CHECK та менше I/O. Агрегація по JSONB (13.4 ms, 1112 buffers) програла нормалізованій таблиці (4.1 ms, 94 buffers).

5. **Новий тип refund.** У JSONB достатньо вставляти рядки з event_type = 'refund' і новою структурою payload, міграція схеми не потрібна. У нормалізованому підході треба створити таблицю t7_hw_refunds_norm з типами та CHECK, заповнити її через INSERT ... SELECT, додати індекси та оновити ETL і запити.

In [ ]:
# engine.dispose()
# db.cleanup()
# print("Сервер зупинено.")